# Supervised Fine-Tuning (SFT)

**Notebook 03 of 7 — LLM Alignment Series**

Supervised Fine-Tuning is the foundation of alignment. While pretraining teaches a model the structure of language and fills it with world knowledge, it does not teach the model *how to behave*. An unaligned pretrained model will happily complete any text sequence — it has no concept of being helpful, following instructions, or refusing harmful requests.

SFT bridges this gap by teaching the model to follow instructions through demonstration. We show the model thousands of examples of high-quality conversations — a human asks a question, and an assistant provides a helpful, well-structured response — and the model learns to imitate that behavior. The training objective is the same as pretraining (next-token prediction), but the *data* is curated to demonstrate the kind of behavior we want.

In the alignment pipeline, SFT typically comes right after pretraining and before reinforcement learning from human feedback (RLHF). It establishes the baseline behavior that RLHF will later refine.

In this notebook we will:
1. Prepare instruction-following data from OpenAssistant
2. Load Qwen2.5-1.5B-Instruct with QLoRA (4-bit quantization + LoRA adapters)
3. Fine-tune the model using TRL's `SFTTrainer`
4. Compare the base model's outputs against the fine-tuned model

---
## Setup

In [ ]:
import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
    TaskType,
)
import bitsandbytes as bnb
import matplotlib.pyplot as plt
import os
import json
from collections import defaultdict

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Why SFT?

A pretrained language model is essentially a powerful text completion engine. Give it a prompt and it will continue the text in a statistically plausible way. But "statistically plausible" is not the same as "helpful and safe." A pretrained model might:

- Continue a question with more questions instead of an answer
- Generate toxic or harmful content if that appeared in training data
- Produce rambling, unfocused text rather than concise responses
- Ignore the user's intent entirely

**SFT gives the model behavior, not just knowledge.** By training on curated instruction-response pairs, the model learns:

- **Turn-taking**: When to stop generating and yield to the user
- **Response format**: How to structure helpful answers (lists, code blocks, explanations)
- **Instruction following**: How to interpret and execute user requests
- **Tone and style**: How to sound like a helpful assistant rather than an autocomplete engine

The training objective remains next-token prediction — the same as pretraining — but the distribution of training data is radically different. Instead of web crawls and books, we use carefully curated conversations that demonstrate ideal assistant behavior.

### SFT in the Alignment Pipeline

```
Pretraining  -->  SFT  -->  Reward Modeling  -->  RLHF/DPO
(knowledge)    (behavior)    (preferences)      (optimization)
```

SFT is not the final step. The model after SFT will be competent but not perfectly aligned — it may still hallucinate, be overly verbose, or occasionally produce unhelpful responses. That is where reward modeling and RLHF come in (Notebooks 04-05). But without SFT, those later stages have nothing useful to build on.

---
## LoRA & QLoRA Explained

### The Problem with Full Fine-Tuning

Full fine-tuning updates every parameter in the model. For Qwen2.5-1.5B that means ~1.5 billion parameters, each requiring gradient storage and optimizer states. In fp32, this would need roughly:

- Model weights: ~6 GB
- Gradients: ~6 GB
- Optimizer states (Adam): ~12 GB
- **Total: ~24 GB** — and that is *before* activations and batch data

This barely fits on a single RTX 4090 even for a 1.5B model, and is completely impractical for larger models.

### LoRA: Low-Rank Adaptation

LoRA (Hu et al., 2021) is based on a key insight: the weight updates during fine-tuning have low intrinsic rank. Instead of updating a full weight matrix $W \in \mathbb{R}^{d \times d}$, LoRA freezes $W$ and adds a low-rank decomposition:

$$W' = W + \Delta W = W + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times d}$, with rank $r \ll d$.

For a layer with $d = 1536$, full fine-tuning updates $1536 \times 1536 = 2.36M$ parameters. With LoRA at $r = 16$, we only update $1536 \times 16 + 16 \times 1536 = 49K$ parameters — a **48x reduction**.

### QLoRA: Quantization + LoRA

QLoRA (Dettmers et al., 2023) goes further by quantizing the frozen base model to 4-bit precision (NF4 — a data type optimized for normally distributed weights). This slashes the memory footprint of the base model from ~6 GB to ~1.5 GB. The LoRA adapters are still trained in fp16/bf16 for numerical stability.

The combination means we can fine-tune a 1.5B parameter model in under 6 GB of VRAM — comfortably within our RTX 4090's 24 GB budget.

### Key LoRA Parameters

| Parameter | Description | Our Value |
|-----------|-------------|----------|
| `r` | Rank of the low-rank matrices. Higher = more capacity but more parameters | 16 |
| `lora_alpha` | Scaling factor. The update is scaled by `alpha/r`. Higher alpha = larger updates | 32 |
| `lora_dropout` | Dropout on the LoRA layers for regularization | 0.05 |
| `target_modules` | Which layers to apply LoRA to. We target all attention projections and the MLP gate/up/down projections | See code below |

---
## Load and Prepare Data

In [ ]:
# Load the OpenAssistant Conversations dataset
raw_dataset = load_dataset("OpenAssistant/oasst1")
print(f"Train split: {len(raw_dataset['train'])} messages")
print(f"Validation split: {len(raw_dataset['validation'])} messages")
print(f"\nColumns: {raw_dataset['train'].column_names}")
print(f"\nExample message:")
print(json.dumps(raw_dataset['train'][0], indent=2, default=str))

In [ ]:
# Combine all messages and build conversation threads
all_messages = list(raw_dataset["train"]) + list(raw_dataset["validation"])

# Create lookup structures
msg_by_id = {msg["message_id"]: msg for msg in all_messages}
children_by_parent = defaultdict(list)
for msg in all_messages:
    if msg["parent_id"] is not None:
        children_by_parent[msg["parent_id"]].append(msg)

# Filter for English, top-ranked responses
# Find root messages (prompter messages with no parent)
root_messages = [
    msg for msg in all_messages
    if msg["parent_id"] is None
    and msg["lang"] == "en"
    and msg["role"] == "prompter"
]
print(f"English root prompts: {len(root_messages)}")


def build_conversation(root_msg, msg_by_id, children_by_parent, max_depth=6):
    """Build a conversation thread by following the highest-ranked child at each step."""
    conversation = []
    current = root_msg

    for _ in range(max_depth):
        role = "user" if current["role"] == "prompter" else "assistant"
        conversation.append({"role": role, "content": current["text"]})

        # Get children and pick the highest-ranked one
        children = children_by_parent.get(current["message_id"], [])
        # Filter for English children with rank information
        english_children = [c for c in children if c.get("lang") == "en"]
        if not english_children:
            break

        # Pick the child with the best (lowest) rank; fall back to first child
        ranked = [c for c in english_children if c.get("rank") is not None]
        if ranked:
            current = min(ranked, key=lambda c: c["rank"])
        else:
            current = english_children[0]

    return conversation


# Build all conversation threads
conversations = []
for root in root_messages:
    conv = build_conversation(root, msg_by_id, children_by_parent)
    # Only keep conversations that have at least a prompt and a response
    if len(conv) >= 2:
        conversations.append(conv)

print(f"Conversations with at least 1 turn: {len(conversations)}")
print(f"Average turns per conversation: {sum(len(c) for c in conversations) / len(conversations):.1f}")

In [ ]:
# Load the tokenizer so we can apply the chat template
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_conversation(conversation):
    """Format a conversation list into a string using the Qwen chat template."""
    return tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=False
    )


# Format all conversations
formatted = [format_conversation(conv) for conv in conversations]

# Create dataset
full_dataset = Dataset.from_dict({"text": formatted})

# Split into train/val (90/10)
split = full_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"Train examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

# Show token length stats
lengths = [len(tokenizer.encode(t)) for t in formatted]
print(f"\nToken length stats:")
print(f"  Min: {min(lengths)}")
print(f"  Max: {max(lengths)}")
print(f"  Mean: {sum(lengths) / len(lengths):.0f}")
print(f"  Median: {sorted(lengths)[len(lengths) // 2]}")

print(f"\n{'='*60}")
print("Example formatted conversation:")
print(f"{'='*60}")
print(train_dataset[0]["text"][:1500])
print("...")

---
## Load Model with QLoRA

In [ ]:
# 4-bit quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the base model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for k-bit training (freeze base, enable gradient checkpointing)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model dtype: {model.dtype}")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print parameter summary
model.print_trainable_parameters()

---
## Configure Training

In [ ]:
# Training configuration
sft_config = SFTConfig(
    output_dir="./results/sft",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=42,
)

# Create the SFT trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print(f"Training batches per epoch: {len(trainer.get_train_dataloader())}")
print(f"Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"Total optimization steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'will be computed at train time'}")

---
## Train

In [ ]:
# Run training
train_result = trainer.train()

# Print summary
print(f"\nTraining completed!")
print(f"  Total steps: {trainer.state.global_step}")
print(f"  Final training loss: {train_result.training_loss:.4f}")

In [ ]:
# Plot training loss
log_history = trainer.state.log_history

train_steps = [entry["step"] for entry in log_history if "loss" in entry]
train_losses = [entry["loss"] for entry in log_history if "loss" in entry]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label="Training Loss", color="steelblue", linewidth=1.5)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("SFT Training Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print eval loss if available
eval_entries = [entry for entry in log_history if "eval_loss" in entry]
if eval_entries:
    for entry in eval_entries:
        print(f"  Step {entry['step']}: eval_loss = {entry['eval_loss']:.4f}")

---
## Save the Adapter

In [ ]:
# Save the LoRA adapter and tokenizer
save_path = "./results/sft/final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to {save_path}")
print(f"\nSaved files:")
for f in sorted(os.listdir(save_path)):
    size_mb = os.path.getsize(os.path.join(save_path, f)) / 1e6
    print(f"  {f:40s} {size_mb:8.2f} MB")

---
## Compare Outputs

Let's compare the base Qwen2.5-1.5B-Instruct model against our SFT-tuned version. Note that the base model is already instruction-tuned by Alibaba — so we are comparing their instruction tuning against ours (fine-tuned on OpenAssistant data).

In [ ]:
# Free memory from the trainer
del trainer
torch.cuda.empty_cache()

# Load the base model fresh (without LoRA)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

# Load the SFT model (base + LoRA adapter)
sft_model = PeftModel.from_pretrained(base_model, save_path)

print("Both models loaded.")

In [ ]:
# Test prompts
test_prompts = [
    "Explain quantum entanglement to a 10-year-old.",
    "Write a Python function that checks if a number is prime.",
    "What are the ethical implications of AI in healthcare?",
    "Give me a recipe for a quick weeknight pasta dinner.",
]


def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """Generate a response from a model given a prompt."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the new tokens
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip()


# Compare outputs side by side
for prompt in test_prompts:
    print(f"{'='*70}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*70}")

    # Disable adapter for base model response
    with sft_model.disable_adapter():
        base_response = generate_response(sft_model, base_tokenizer, prompt)

    # Enable adapter for SFT model response
    sft_response = generate_response(sft_model, base_tokenizer, prompt)

    print(f"\n--- Base Model ---")
    print(base_response[:500])
    print(f"\n--- SFT Model ---")
    print(sft_response[:500])
    print()

---
## Loading Adapters Later

One of the great advantages of LoRA is that the adapter files are small (typically 10-50 MB) while the base model remains unchanged. This means you can:

- Store multiple fine-tuned variants without duplicating the base model
- Quickly swap between different fine-tunes at inference time
- Share adapters without sharing the full model weights

Here is how to reload a saved adapter and optionally merge it into the base model.

In [ ]:
# Method 1: Load adapter on top of base model (keeps them separate)
# This is useful when you want to swap adapters or disable them at runtime
adapter_path = "./results/sft/final"

reloaded_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
reloaded_model = PeftModel.from_pretrained(reloaded_base, adapter_path)
print("Adapter loaded on top of base model.")
print(f"Active adapters: {reloaded_model.active_adapters}")

# Quick test
test_response = generate_response(
    reloaded_model, base_tokenizer, "What is machine learning?"
)
print(f"\nTest response: {test_response[:300]}")

In [ ]:
# Method 2: Merge adapter into base model (creates a standalone model)
# Note: merge_and_unload requires the model NOT be quantized.
# We demonstrate the API here; for a quantized model you would first
# load the base in fp16/bf16, then merge.

# Load in fp16 for merging (requires more VRAM)
merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
merge_model = PeftModel.from_pretrained(merge_base, adapter_path)

# Merge LoRA weights into the base model
merged_model = merge_model.merge_and_unload()
print(f"Merged model type: {type(merged_model).__name__}")
print("The model is now a standard transformers model with LoRA weights baked in.")

# You could save the full merged model:
# merged_model.save_pretrained("./results/sft/merged")
# base_tokenizer.save_pretrained("./results/sft/merged")

# Clean up to free memory
del reloaded_base, reloaded_model, merge_base, merge_model, merged_model
torch.cuda.empty_cache()
print("\nMemory cleaned up.")

---
## Summary & Next Steps

### What We Did

In this notebook we performed Supervised Fine-Tuning on Qwen2.5-1.5B-Instruct:

1. **Prepared data** from the OpenAssistant OASST1 dataset — filtering for English, top-ranked responses, and building multi-turn conversation threads formatted with the Qwen chat template.

2. **Applied QLoRA** — loading the base model in 4-bit NF4 quantization and attaching LoRA adapters (rank 16) to all attention and MLP projection layers. This reduced trainable parameters from ~1.5B to a few million while keeping memory usage well within our 24 GB VRAM budget.

3. **Trained with SFTTrainer** from the TRL library for one epoch, using paged AdamW in 8-bit with cosine learning rate scheduling.

4. **Compared outputs** between the base model and the fine-tuned model on diverse test prompts to observe the effect of SFT.

5. **Saved and reloaded** the LoRA adapter, demonstrating both the separate-adapter workflow and the merge-and-unload approach for creating a standalone model.

### Key Takeaways

- SFT teaches the model *how to behave* rather than *what to know*. The training objective (next-token prediction) is the same as pretraining, but the curated data shapes the model's response style and instruction-following ability.
- QLoRA makes it practical to fine-tune on a single consumer GPU by quantizing the frozen base model and only training small low-rank adapter matrices.
- The quality of SFT depends heavily on the quality of the training data. Garbage in, garbage out.

### Next: Notebook 04 — Reward Modeling

SFT gets us a model that can follow instructions, but it has no way to distinguish between good and bad responses of its own. In the next notebook, we will train a **reward model** — a model that scores responses by quality. This reward model will later be used to optimize our SFT model via RLHF (Notebook 05) or DPO (Notebook 06).